In [11]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use('Agg')  # non-interactive backend, safe for batch processing
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from scipy.signal import butter, filtfilt
from scipy.ndimage import gaussian_filter1d

from config import dir_config, ephys_config

In [12]:
compiled_dir = Path(dir_config.data.compiled)
sorted_dir = Path(dir_config.data.sorting)
processed_dir = Path(dir_config.data.processed)

### Helper Function

#### Loading data

In [13]:
def load_phy_data(session_id):
    kl_dir = Path(sorted_dir, session_id, "kilosort4")
    sorting_data = {
        "spike_times": np.load(kl_dir / "spike_times.npy").flatten(),
        "spike_clusters": np.load(kl_dir / "spike_clusters.npy").flatten(),
        "amplitudes": np.load(kl_dir / "amplitudes.npy").flatten(),
        "templates": np.load(kl_dir / "templates.npy"),  # (n_units, n_samples, n_channels)
        "chan_pos": np.load(kl_dir / "channel_positions.npy"),  # (n_channels, 2)
    }

    winv_path = kl_dir / 'whitening_mat_inv.npy'
    sorting_data['whitening_mat_inv'] = np.load(winv_path) if winv_path.exists() else None

    # Phy label file: prefer manually curated cluster_group.tsv
    group_path = kl_dir / 'cluster_group.tsv'
    ks_path    = kl_dir / 'cluster_KSLabel.tsv'
    if group_path.exists():
        sorting_data['cluster_groups'] = pd.read_csv(group_path, sep='\t')
    elif ks_path.exists():
        sorting_data['cluster_groups'] = pd.read_csv(ks_path, sep='\t')
    else:
        sorting_data['cluster_groups'] = None

    # Sample rate and recording parameters from params.py
    params_path = kl_dir / 'params.py'
    params = {}
    if params_path.exists():
        with open(params_path) as f:
            exec(f.read(), params)
    sorting_data['sample_rate'] = int(params.get('sample_rate', 30000))
    sorting_data['n_channels']  = int(params.get('n_channels_dat', sorting_data['templates'].shape[2]))
    sorting_data['dtype']       = params.get('dtype', 'int16')

    return sorting_data

def load_trial_events(session_id, sample_rate=30000, valid_only=True):
    filename = f"{session_id}_timestamps_cleaned.csv"
    filename = f"{session_id}_timestamps.csv"
    timestamps = pd.read_csv(Path(compiled_dir, session_id, filename), index_col=None)
    if valid_only:
        timestamps = timestamps[~np.isnan(timestamps['response_onset'])]


    event_cols = [
    'fixation_onset', 'target_onset', 'stimulus_onset',
    'go_onset', 'response_onset', 'trial_offset',
    ]
    events = {}
    for col in event_cols:
        if col in timestamps.columns:
            events[col] = pd.to_numeric(timestamps[col]) / sample_rate  # samples -> seconds
        else:
            print(f"  Warning: column '{col}' not found in trial file, skipping.")
    return events

def get_good_units(cluster_groups, label='good'):
    """Return cluster IDs labelled as 'good' in Phy."""
    if cluster_groups is None:
        return None
    col = 'group' if 'group' in cluster_groups.columns else cluster_groups.columns[-1]
    mask = cluster_groups[col].str.lower() == label.lower()
    return cluster_groups.loc[mask, 'cluster_id'].values


In [14]:
session_id = "250109_GP_TZ"

sorting_data = load_phy_data(session_id)
events = load_trial_events(session_id, sample_rate=sorting_data['sample_rate'], valid_only=True)
good_units = get_good_units(sorting_data['cluster_groups'], label='good')

#### Quality Metrics

In [15]:
def compute_fr_similarity(spike_times_sec, trial_times_sec, before=2.0):
    """
    Firing rate stability score anchored to fixation_onset.
    Splits trials in half, computes mean FR in the pre-fixation window for each half.

    Score = 1 - 2 * |FR1 - FR2| / (FR1 + FR2)
      1   = perfectly stable across the session
      0   = FR in one half is double the other
     <0   = severe drift (one half near-silent)
     NaN  = unit silent in at least one half
    """

    valid = trial_times_sec[~np.isnan(trial_times_sec)]

    if len(valid) < 2:
        return np.nan  # not enough trials to compute stability

    half = len(valid) // 2

    def mean_fr(trial_subset):
        counts = [
            np.sum((spike_times_sec >= t - before) & (spike_times_sec < t))
            for t in trial_subset
        ]
        return np.mean(counts)

    fr1 = mean_fr(valid[:half])
    fr2 = mean_fr(valid[half:])
    denom = fr1 + fr2
    if denom == 0:
        return np.nan  # unit silent in at least one half
    score = 1 - 2 * abs(fr1 - fr2) / denom
    return score

def compute_isi_violations(spike_times_sec, refractory_period_ms=1.5):
    """
    Refractory period violation ratio.
    Returns fraction of ISIs below the refractory period threshold.
    Pass threshold: < 2%.
    """
    if len(spike_times_sec) < 2:
        return np.nan
    isis = np.diff(np.sort(spike_times_sec))
    rp_s = refractory_period_ms / 1000.0
    return np.sum(isis < rp_s) / len(isis)

def compute_noise_cutoff(amplitudes, n_bins=100, percent_threshold=10.0):
    """
    Noise cutoff metric (IBL pipeline).
    Checks if the amplitude histogram is truncated at the low end.
    metric_value = lowest bin count as % of peak bin count.
    Pass if metric_value < percent_threshold (default 10%).
    """
    if len(amplitudes) < 10:
        return False, np.nan
    counts, _ = np.histogram(amplitudes, bins=n_bins)
    if counts.max() == 0:
        return False, np.nan
    low_bin_pct = counts[0] / counts.max() * 100
    return low_bin_pct < percent_threshold, low_bin_pct

def compute_acg(spike_times_sec, bin_ms=0.5, max_lag_ms=50):
    """
    Autocorrelogram. Returns (bin_centers_ms, counts).
    """
    if len(spike_times_sec) < 2:
        return np.array([]), np.array([])

    bin_s   = bin_ms / 1000.0
    max_lag = max_lag_ms / 1000.0
    bins    = np.arange(-max_lag, max_lag + bin_s, bin_s)
    counts  = np.zeros(len(bins) - 1)
    st      = np.sort(spike_times_sec)

    for i, t in enumerate(st):
        diffs = st - t
        diffs = diffs[(diffs != 0) & (np.abs(diffs) <= max_lag)]
        counts += np.histogram(diffs, bins=bins)[0]

    bin_centers = (bins[:-1] + bins[1:]) / 2 * 1000  # convert to ms
    return bin_centers, counts


def sliding_rp_confidence(spike_times_sec, sample_rate, rp_range_ms=None, cont_range=None):
    """
    Sliding refractory period confidence matrix (simplified IBL/Llobet approach).
    Returns (conf_matrix, rp_range_ms, cont_range).
    conf_matrix shape: (n_cont, n_rp), values 0-100.
    """
    from scipy.stats import poisson

    if rp_range_ms is None:
        rp_range_ms = np.arange(0.5, 5.1, 0.5)
    if cont_range is None:
        cont_range = np.arange(0, 31, 1)

    n_spikes = len(spike_times_sec)
    if n_spikes < 2:
        return np.zeros((len(cont_range), len(rp_range_ms))), rp_range_ms, cont_range

    duration    = spike_times_sec.max() - spike_times_sec.min()
    firing_rate = n_spikes / duration if duration > 0 else 0
    isis        = np.diff(np.sort(spike_times_sec))

    conf_matrix = np.zeros((len(cont_range), len(rp_range_ms)))
    for i, rp_ms in enumerate(rp_range_ms):
        rp_s   = rp_ms / 1000.0
        n_viol = np.sum(isis < rp_s)
        for j, cont in enumerate(cont_range):
            expected = (cont / 100.0) * firing_rate * rp_s * n_spikes * 2
            if expected == 0:
                conf_matrix[j, i] = 100.0 if n_viol == 0 else 0.0
            else:
                conf_matrix[j, i] = poisson.cdf(n_viol, expected) * 100

    return conf_matrix, rp_range_ms, cont_range


#### Waveform Extraction from raw binary data

In [16]:
def extract_waveforms(session_id, spike_times_samples, n_channels, sample_rate,
                      dtype='int16', n_wf=200, hp_cutoff=300):
    """
    Extract and high-pass filter waveforms from raw binary file.
    Returns array of shape (n_wf, win_samples, n_channels), or None if unavailable.
    """
    raw_path = Path(sorted_dir, session_id, f"{session_id}.bin")

    if not raw_path.exists():
        print(f"  Warning: raw data file not found: {raw_path}")
        return None

    win_samples = int(sample_rate / 1000) * 2 + 1  # +/- 1 ms
    half        = win_samples // 2
    dt          = np.dtype(dtype)

    n_spikes = len(spike_times_samples)
    idx      = np.random.choice(n_spikes, min(n_wf, n_spikes), replace=False)
    selected = np.sort(spike_times_samples[idx])

    waveforms = []
    with open(raw_path, 'rb') as f:
        for t in selected:
            start = int(t) - half
            if start < 0:
                continue
            f.seek(start * n_channels * dt.itemsize)
            chunk = np.frombuffer(
                f.read(win_samples * n_channels * dt.itemsize), dtype=dt
            )
            if len(chunk) != win_samples * n_channels:
                continue
            waveforms.append(chunk.reshape(win_samples, n_channels))

    if not waveforms:
        return None

    wf_array = np.array(waveforms, dtype=float)  # (n_wf, win_samples, n_channels)

    b, a = butter(2, hp_cutoff / (sample_rate / 2), btype='high', output='ba')
    for w in range(wf_array.shape[0]):
        for ch in range(n_channels):
            wf_array[w, :, ch] = filtfilt(b, a, wf_array[w, :, ch])

    return wf_array


def get_best_channels(wf_array, chan_pos, n=4):
    """Return indices of the n channels with highest peak-to-peak amplitude."""
    mean_wf = wf_array.mean(axis=0)                             # (win_samples, n_channels)
    ptp     = mean_wf.max(axis=0) - mean_wf.min(axis=0)
    best_ch = np.argmax(ptp)
    d       = np.sqrt(((chan_pos - chan_pos[best_ch]) ** 2).sum(axis=1))
    d[best_ch] = np.inf
    nearest    = np.argsort(d)[:n - 1]
    return np.concatenate([[best_ch], nearest])


#### Plotting

In [17]:

PASS_COLOR = np.array([34, 177, 76]) / 255
FAIL_COLOR = np.array([220, 50, 50]) / 255
NAN_COLOR  = np.array([200, 180, 0]) / 255

RASTER_EVENTS = [
    ('fixation_onset', 'Fixation onset'),
    ('stimulus_onset',       'Go signal'),
    ('response_onset', 'Response onset'),
]


def fmt_pass(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '~'
    return 'PASS' if val else 'FAIL'


def unit_color(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return NAN_COLOR
    return PASS_COLOR if val else FAIL_COLOR


def plot_raster_panel(ax, spike_times_sec, event_times_sec, before, after,
                      event_label, fr_similarity=None):
    # CHANGE 1: cast inputs to float at entry so all arithmetic is unambiguous
    spike_times_sec  = np.asarray(spike_times_sec,  dtype=float)
    event_times_sec  = np.asarray(event_times_sec,  dtype=float)

    valid_mask  = ~np.isnan(event_times_sec)
    valid_times = event_times_sec[valid_mask]
    n_trials    = len(valid_times)

    if n_trials == 0:
        ax.text(0.5, 0.5, f'No valid trials\nfor {event_label}',
                ha='center', va='center', transform=ax.transAxes, color='gray')
        ax.set_title(event_label, fontsize=9)
        return

    bin_s  = 0.01
    bins   = np.arange(-before, after + bin_s, bin_s)
    n_bins = len(bins) - 1
    psth   = np.zeros((n_trials, n_bins))

    for i, t0 in enumerate(valid_times):
        rel = spike_times_sec - t0
        spk = rel[(rel >= -before) & (rel < after)]
        if len(spk):
            ax.vlines(spk, i + 0.5, i + 1.5,
                      color='black', linewidth=0.4, alpha=0.7)
        psth[i], _ = np.histogram(rel, bins=bins)
        # CHANGE 2: explicit cast avoids the operator error on ambiguous array type
        psth[i] = psth[i].astype(float) / bin_s  # Hz

    ax.axvline(0, color='cyan', linewidth=1.2, linestyle='--')
    ax.set_xlim(-before, after)
    ax.set_ylim(0, n_trials)
    ax.set_xlabel('Time (s)', fontsize=8)
    ax.set_ylabel('Trial #', fontsize=8)
    ax.tick_params(labelsize=7)

    mean_fr = gaussian_filter1d(psth.mean(axis=0), sigma=3)
    t_axis  = (bins[:-1] + bins[1:]) / 2
    ax2 = ax.twinx()
    ax2.plot(t_axis, mean_fr, color='royalblue', linewidth=1.2, alpha=0.6)
    ax2.set_ylabel('Mean FR (Hz)', color='royalblue', fontsize=7)
    ax2.tick_params(axis='y', colors='royalblue', labelsize=7)

    title_str = f'{event_label}  ({n_trials} trials)'
    if fr_similarity is not None:
        title_str += f'  |  stability = {fr_similarity:.3f}'
    ax.set_title(title_str, fontsize=9)


def plot_unit(session_id, unit_idx, cluster_id, spike_times_sec, amplitudes,
              trial_events, fr_similarity, isi_ratio, nc_pass, nc_value,
              acg_bins, acg_counts, conf_matrix, rp_range_ms, cont_range,
              wf_array, chan_pos,
              before=2.0, after=2.0, n_units_total=1, output_dir=None):

    # CHANGE 3: cast inputs to float at entry so all downstream arithmetic is clean
    spike_times_sec = np.asarray(spike_times_sec, dtype=float)
    amplitudes      = np.asarray(amplitudes,      dtype=float)

    stable   = (not np.isnan(fr_similarity)) and fr_similarity > 0
    isi_pass = (not np.isnan(isi_ratio)) and isi_ratio < 0.02
    overall  = stable and isi_pass and nc_pass

    fig = plt.figure(figsize=(24, 14), facecolor='white')
    gs  = gridspec.GridSpec(3, 5, figure=fig, hspace=0.55, wspace=0.42)

    stab_str = ('Stable' if stable
                else ('NaN FR' if np.isnan(fr_similarity) else 'Unstable'))
    line1 = (f"Session {session_id}  |  Unit {unit_idx + 1} of {n_units_total}  |  Cluster {cluster_id}  |  "
             f"Overall: {fmt_pass(overall)}")
    line2 = (f"Stability: {fmt_pass(stable)} ({stab_str})  |  "
             f"ISI violations: {fmt_pass(isi_pass)} ({isi_ratio * 100:.2f}%)  |  "
             f"Noise cutoff: {fmt_pass(nc_pass)} ({nc_value:.1f}%)")
    gt = fig.suptitle(f"{line1}\n{line2}", fontsize=11, fontweight='bold', y=0.99)
    gt.set_color(unit_color(overall))

    for col_i, (event_key, event_label) in enumerate(RASTER_EVENTS):
        ax = fig.add_subplot(gs[0, col_i])
        if event_key in trial_events:
            sim_label = fr_similarity if col_i == 0 else None
            plot_raster_panel(ax, spike_times_sec, trial_events[event_key],
                              before, after, event_label, sim_label)
        else:
            ax.text(0.5, 0.5, f'{event_label}\nnot in trial file',
                    ha='center', va='center', transform=ax.transAxes, color='gray')
            ax.set_title(event_label, fontsize=9)

    ax_frt    = fig.add_subplot(gs[0, 3])
    fix_times = trial_events.get('fixation_onset', np.array([]))
    valid_fix = fix_times[~np.isnan(fix_times)] if len(fix_times) else np.array([])

    if len(valid_fix) > 0:
        trial_fr = np.array([
            np.sum((spike_times_sec >= t0 - before) & (spike_times_sec < t0)) / before
            for t0 in valid_fix
        ])
        ax_frt.plot(trial_fr, 'k.', markersize=3, alpha=0.5)
        win     = min(20, len(trial_fr))
        mov_avg = np.convolve(trial_fr, np.ones(win) / win, mode='valid')
        ax_frt.plot(np.arange(win // 2, win // 2 + len(mov_avg)), mov_avg,
                    color='royalblue', linewidth=1.8)
        ax_frt.axhline(trial_fr.mean(), color='royalblue', linestyle='--', linewidth=1)
        ax_frt.axvline(len(trial_fr) // 2, color='gray', linestyle=':', linewidth=1)

    ax_frt.set_xlabel('Trial number', fontsize=8)
    ax_frt.set_ylabel('Mean FR (Hz)', fontsize=8)
    ax_frt.set_title(f'FR across trials\n({before}s pre-fixation window)', fontsize=9)
    ax_frt.tick_params(labelsize=7)

    ax_sum = fig.add_subplot(gs[0, 4])
    ax_sum.axis('off')
    summary_text = (
        f"Cluster {cluster_id}\n"
        f"N spikes: {len(spike_times_sec):,}\n"
        f"Mean FR:  {len(spike_times_sec) / (spike_times_sec.max() - spike_times_sec.min()):.2f} Hz\n\n"
        f"FR similarity:  {fr_similarity:.3f}\n"
        f"ISI viol:       {isi_ratio * 100:.2f}%\n"
        f"Noise cutoff:   {nc_value:.1f}%\n\n"
        f"Stability:  {fmt_pass(stable)}\n"
        f"ISI:        {fmt_pass(isi_pass)}\n"
        f"Noise cut:  {fmt_pass(nc_pass)}\n\n"
        f"OVERALL:    {fmt_pass(overall)}"
    )
    ax_sum.text(0.05, 0.95, summary_text, transform=ax_sum.transAxes,
                fontsize=9, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))

    ax_amp = fig.add_subplot(gs[1, 0])
    ax_amp.scatter(spike_times_sec / 60, amplitudes, s=1, c='black', alpha=0.25)
    ax_amp.set_xlabel('Time in session (min)', fontsize=8)
    ax_amp.set_ylabel('Template amplitude (a.u.)', fontsize=8)
    ax_amp.set_title('Spike amplitudes over time', fontsize=9)
    ax_amp.tick_params(labelsize=7)

    ax_hist = fig.add_subplot(gs[1, 1])
    ax_hist.hist(amplitudes, bins=100, orientation='horizontal',
                 color='steelblue', edgecolor='none', alpha=0.85)
    ax_hist.set_xlabel('Count', fontsize=8)
    ax_hist.set_ylabel('Amplitude (a.u.)', fontsize=8)
    nc_col = PASS_COLOR if nc_pass else FAIL_COLOR
    ax_hist.set_title(f'Amplitude histogram\nLow bin: {nc_value:.1f}% of peak',
                      color=nc_col, fontsize=9)
    ax_hist.tick_params(labelsize=7)

    ax_acg = fig.add_subplot(gs[2, 0])
    if len(acg_bins) > 0:
        ax_acg.bar(acg_bins, acg_counts, width=(acg_bins[1] - acg_bins[0]),
                   color='steelblue', edgecolor='none')
        ax_acg.axvspan(-1.5, 1.5, color='red', alpha=0.15)
    ax_acg.set_xlabel('Lag (ms)', fontsize=8)
    ax_acg.set_ylabel('Count', fontsize=8)
    ax_acg.set_title(f'Autocorrelogram\nISI violations: {isi_ratio * 100:.2f}%',
                     color=unit_color(isi_pass), fontsize=9)
    ax_acg.tick_params(labelsize=7)

    ax_rp = fig.add_subplot(gs[2, 1])
    cmap  = LinearSegmentedColormap.from_list(
        'conf', ['#1a1a2e', '#16213e', '#0f3460', '#53d8fb']
    )
    im = ax_rp.imshow(conf_matrix, aspect='auto', origin='upper', cmap=cmap,
                      vmin=0, vmax=100,
                      extent=[rp_range_ms[0], rp_range_ms[-1],
                               cont_range[-1], cont_range[0]])
    ax_rp.axhline(10, color='red', linewidth=1)
    plt.colorbar(im, ax=ax_rp, label='Confidence (%)')
    ax_rp.set_xlabel('Refractory period (ms)', fontsize=8)
    ax_rp.set_ylabel('Contamination (%)', fontsize=8)
    ax_rp.set_title('Sliding RP confidence', fontsize=9)
    ax_rp.tick_params(labelsize=7)

    wf_positions = [(1, 2), (1, 3), (2, 2), (2, 3)]

    if wf_array is not None:
        best_chs = get_best_channels(wf_array, chan_pos, n=4)
        t_wf     = np.linspace(-1, 1, wf_array.shape[1])

        for plot_i, ch in enumerate(best_chs):
            r, c  = wf_positions[plot_i]
            ax_wf = fig.add_subplot(gs[r, c])
            wf_ch = wf_array[:, :, ch]

            ax_wf.plot(t_wf, wf_ch.T, color='gray', linewidth=0.3, alpha=0.25)
            ax_wf.plot(t_wf, wf_ch.mean(axis=0), color='black', linewidth=1.8)
            ax_wf.axvline(0, color='red', linewidth=0.8, linestyle='--')

            d     = np.sqrt(((chan_pos[ch] - chan_pos[best_chs[0]]) ** 2).sum())
            label = f'Ch {ch}' if plot_i == 0 else f'Ch {ch}  ({d:.0f} um away)'
            ax_wf.set_title(label, fontsize=8)
            ax_wf.set_xlabel('Time (ms)', fontsize=7)
            ax_wf.tick_params(labelsize=7)
    else:
        for plot_i in range(4):
            r, c  = wf_positions[plot_i]
            ax_wf = fig.add_subplot(gs[r, c])
            ax_wf.text(0.5, 0.5,
                       'Raw data not provided\n(pass --raw_data to enable)',
                       ha='center', va='center', fontsize=8, color='gray',
                       transform=ax_wf.transAxes)
            ax_wf.axis('off')

    plt.show()
    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        base = os.path.join(output_dir, f'Unit_{unit_idx + 1:03d}_Cluster_{cluster_id}')
        fig.savefig(base + '.png', dpi=150, bbox_inches='tight')
        fig.savefig(base + '.pdf', bbox_inches='tight')

    # plt.close(fig)

## Main Code

In [ ]:
def run_qc(session_id, output_dir=None, before=2.0, after=2.0, good_only=True):

    print(f"Loading Phy data from: {session_id}")
    d  = load_phy_data(session_id)
    fs = d['sample_rate']
    print(f"  Sample rate: {fs} Hz")

    print(f"Loading trial events from: {session_id}")
    trial_events = load_trial_events(session_id, fs)
    print(f"  Events loaded: {list(trial_events.keys())}")
    for k, v in trial_events.items():
        n_valid = int(np.sum(~np.isnan(v)))
        print(f"    {k}: {n_valid} valid trials out of {len(v)}")

    spike_times_sec = d['spike_times'] / fs

    # Select units to process
    if good_only and d['cluster_groups'] is not None:
        cluster_ids = get_good_units(d['cluster_groups'], label='good')
        if cluster_ids is None or len(cluster_ids) == 0:
            print("  No 'good' units found, processing all units.")
            cluster_ids = np.unique(d['spike_clusters'])
    else:
        cluster_ids = np.unique(d['spike_clusters'])

    print(f"  Processing {len(cluster_ids)} units\n")

    summary_rows = []

    for unit_idx, cluster_id in enumerate(cluster_ids):
        print(f"  Unit {unit_idx + 1}/{len(cluster_ids)}  Cluster {cluster_id}", end='  ')

        mask = d['spike_clusters'] == cluster_id
        st   = spike_times_sec[mask]
        amps = d['amplitudes'][mask]

        if len(st) < 10:
            print("SKIP (fewer than 10 spikes)")
            continue

        fix_times = trial_events.get('fixation_onset', np.array([np.nan]))
        fr_sim    = compute_fr_similarity(st, fix_times, before=before)
        isi_r     = compute_isi_violations(st)
        nc_pass, nc_val = compute_noise_cutoff(amps)
        acg_bins, acg_counts = compute_acg(st)
        conf_mat, rp_ax, cont_ax = sliding_rp_confidence(st, fs)

        wf_array = extract_waveforms(session_id, d['spike_times'][mask], d['n_channels'], fs, dtype=d['dtype'])

        plot_unit(
            session_id=session_id,
            unit_idx=unit_idx,
            cluster_id=cluster_id,
            spike_times_sec=st,
            amplitudes=amps,
            trial_events=trial_events,
            fr_similarity=fr_sim,
            isi_ratio=isi_r,
            nc_pass=nc_pass,
            nc_value=nc_val,
            acg_bins=acg_bins,
            acg_counts=acg_counts,
            conf_matrix=conf_mat,
            rp_range_ms=rp_ax,
            cont_range=cont_ax,
            wf_array=wf_array,
            chan_pos=d['chan_pos'],
            before=before,
            after=after,
            n_units_total=len(cluster_ids),
            output_dir=str(output_dir) if output_dir is not None else None,
        )

        stable = (not np.isnan(fr_sim)) and fr_sim > 0
        isi_ok = (not np.isnan(isi_r)) and isi_r < 0.02
        summary_rows.append({
            'cluster_id':        cluster_id,
            'n_spikes':          len(st),
            'mean_fr_hz':        round(len(st) / (st.max() - st.min()), 2),
            'fr_similarity':     round(fr_sim, 4) if not np.isnan(fr_sim) else np.nan,
            'stability_pass':    stable,
            'isi_violation_pct': round(isi_r * 100, 3) if not np.isnan(isi_r) else np.nan,
            'isi_pass':          isi_ok,
            'noise_cutoff_pct':  round(nc_val, 2) if not np.isnan(nc_val) else np.nan,
            'nc_pass':           nc_pass,
            'overall_pass':      stable and isi_ok and nc_pass,
        })
        print(f"done  [overall: {fmt_pass(stable and isi_ok and nc_pass)}]")

    summary_df = pd.DataFrame(summary_rows)

    if output_dir is not None:
        csv_path   = Path(output_dir) / 'qc_summary.csv'
        summary_df.to_csv(csv_path, index=False)
        print(f"\nDone. {len(summary_rows)} units processed.")
        print(f"Summary CSV : {csv_path}")
        print(f"Figures     : {output_dir}")
    return summary_df


In [ ]:
session_id = "241216_GP_TZ"

run_qc(session_id=session_id, output_dir=None, before=2.0, after=2.0, good_only=True)

Loading Phy data from: 241216_GP_TZ
  Sample rate: 30000 Hz
Loading trial events from: 241216_GP_TZ
  Events loaded: ['fixation_onset', 'target_onset', 'stimulus_onset', 'go_onset', 'response_onset', 'trial_offset']
    fixation_onset: 1254 valid trials out of 1254
    target_onset: 1254 valid trials out of 1254
    stimulus_onset: 1179 valid trials out of 1254
    go_onset: 1254 valid trials out of 1254
    response_onset: 1254 valid trials out of 1254
    trial_offset: 1254 valid trials out of 1254
  Processing 9 units

  Unit 1/9  Cluster 3  